In [1]:
# Install libraries
# datasets < 4.0.0 is required: newer versions dropped support for
# "loading scripts", which is what tner/ontonotes5 is built on

!pip install -q --upgrade pip setuptools wheel
!pip install -q numpy
!pip install -q seqeval
!pip install -q "datasets<4.0.0"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
# Load the dataset
# tner/ontonotes5: a "flat" version of OntoNotes 5.0 (just tokens/tags,
# no nested document structure). 18 entity types, BIO tagging scheme

from datasets import load_dataset
from transformers import AutoTokenizer

ds = load_dataset("tner/ontonotes5")

README.md: 0.00B [00:00, ?B/s]

ontonotes5.py: 0.00B [00:00, ?B/s]

ontonotes5/train/0000.parquet:   0%|          | 0.00/3.68M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/515k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/517k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/59924 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8528 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8262 [00:00<?, ? examples/s]

In [3]:
# Original 37-label list
# Written out manually: the parquet auto-conversion of this dataset
# strips label names. Verified against the official label2id.json
# from the tner project — exact match

label_list = [
    "O", "B-CARDINAL", "B-DATE", "I-DATE", "B-PERSON", "I-PERSON", "B-NORP", "B-GPE", "I-GPE",
    "B-LAW", "I-LAW", "B-ORG", "I-ORG", "B-PERCENT", "I-PERCENT", "B-ORDINAL", "B-MONEY", "I-MONEY",
    "B-WORK_OF_ART", "I-WORK_OF_ART", "B-FAC", "B-TIME", "I-CARDINAL", "B-LOC", "B-QUANTITY",
    "I-QUANTITY", "I-NORP", "I-LOC", "B-PRODUCT", "I-TIME", "B-EVENT", "I-EVENT", "I-FAC",
    "B-LANGUAGE", "I-PRODUCT", "I-ORDINAL", "I-LANGUAGE"
]

In [4]:
# Collapse 37 labels down to 5 (O, B-LOC, I-LOC, B-DATE, I-DATE)
# GPE (cities/countries) and LOC (mountains, rivers, etc.) both map to
# LOC — for the anonymization task the distinction doesn't matter

new_label_list = ["O", "B-LOC", "I-LOC", "B-DATE", "I-DATE"]
new_label2id = {l: i for i, l in enumerate(new_label_list)}
new_id2label = {i: l for i, l in enumerate(new_label_list)}

def remap(old_name: str) -> str:
    if old_name in ("B-GPE", "B-LOC"):
        return "B-LOC"
    if old_name in ("I-GPE", "I-LOC"):
        return "I-LOC"
    if old_name == "B-DATE":
        return "B-DATE"
    if old_name == "I-DATE":
        return "I-DATE"
    return "O"

old_id_to_new_id = {old_id: new_label2id[remap(name)] for old_id, name in enumerate(label_list)}

In [5]:
def remap_tags(example: dict) -> dict:
    example["tags"] = [old_id_to_new_id[t] for t in example["tags"]]
    return example

In [6]:
# Tokenization + label alignment to subwords
# bert-base-cased ("cased" keeps letter casing — an important NER signal).
# Only the first subword of each word gets the real label; the rest
# get -100, which PyTorch ignores when computing loss

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_and_align_labels(example: dict) -> dict:
    tokenized = tokenizer(example["tokens"], is_split_into_words=True, truncation=True)
    word_ids = tokenized.word_ids()
    labels, prev_word_id = [], None
    for word_id in word_ids:
        if word_id is None:
            labels.append(-100)
        elif word_id != prev_word_id:
            labels.append(example["tags"][word_id])
        else:
            labels.append(-100)
        prev_word_id = word_id
    tokenized["labels"] = labels
    return tokenized

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
# Apply preprocessing to the official train/validation/test splits
# (mentor-approved — safer than a self-made split, since this flat
# dataset has no document id to prevent cross-split leakage)

ds = ds.map(remap_tags)
ds = ds.map(tokenize_and_align_labels)

train_ds = ds["train"]
val_ds = ds["validation"]
test_ds = ds["test"]

print(f"train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}")
print(train_ds[0])

Map:   0%|          | 0/59924 [00:00<?, ? examples/s]

Map:   0%|          | 0/8528 [00:00<?, ? examples/s]

Map:   0%|          | 0/8262 [00:00<?, ? examples/s]

Map:   0%|          | 0/59924 [00:00<?, ? examples/s]

Map:   0%|          | 0/8528 [00:00<?, ? examples/s]

Map:   0%|          | 0/8262 [00:00<?, ? examples/s]

train: 59924, val: 8528, test: 8262
{'tokens': ['People', 'start', 'their', 'own', 'businesses', 'for', 'many', 'reasons', '.'], 'tags': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'input_ids': [101, 2563, 1838, 1147, 1319, 5028, 1111, 1242, 3672, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 0, 0, 0, 0, 0, 0, 0, 0, 0, -100]}


In [8]:
import numpy as np
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
from seqeval.metrics import f1_score, precision_score, recall_score, accuracy_score

In [9]:
# Model setup
# Loads pretrained bert-base-cased and attaches a new, randomly
# initialized classification head for our 5 labels — this new head
# is exactly what fine-tuning trains

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(new_label_list),
    id2label=new_id2label,
    label2id=new_label2id,
)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

In [10]:
# Metrics: entity-level F1/precision/recall (main metric, per Тech task + mentor)
# Accuracy is also computed only for comparison — it looks deceptively
# high on NER because the "O" class dominates

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [new_label_list[p] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]
    true_labels = [
        [new_label_list[l] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
        "accuracy": accuracy_score(true_labels, true_predictions),
    }

In [11]:
# Data collator
# Pads sequences to equal length within each batch, since sentences
# have different lengths but the model needs a rectangular tensor

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [12]:
# Training configuration — smoke test: just 1 epoch, to confirm the
# whole pipeline (model -> training -> eval -> metrics) runs end-to-end
# before committing to a full training run

# training_args = TrainingArguments(
#     output_dir="./results",
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     learning_rate=2e-5,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     num_train_epochs=1,
#     weight_decay=0.01,
#     load_best_model_at_end=True,
#     metric_for_best_model="f1",
# )


In [13]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",   # NEW: logs train loss per epoch too, not just eval
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,          # was 1 — now a real training run
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

In [14]:
# Train
# processing_class replaces the older "tokenizer" argument name in
# recent versions of transformers' Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,    
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [15]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.068100,0.044162,0.886051,0.906760,0.896286,0.992784
2,0.029670,0.044426,0.892937,0.899221,0.896068,0.992817
3,0.018759,0.050436,0.888944,0.901231,0.895046,0.992838


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=5619, training_loss=0.038843272016192275, metrics={'train_runtime': 1730.6969, 'train_samples_per_second': 103.873, 'train_steps_per_second': 3.247, 'total_flos': 6198235372007640.0, 'train_loss': 0.038843272016192275, 'epoch': 3.0})

In [16]:
# Experiment 2: same setup, but lower learning_rate (to also test if it reduces
# the mild overfitting seen in epoch 3 of experiment 1)

model_exp2 = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(new_label_list),
    id2label=new_id2label,
    label2id=new_label2id,
)

training_args_exp2 = TrainingArguments(
    output_dir="./results_lr1e-5",   # different folder, so it doesn't overwrite experiment 1
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=1e-5,               # only this changed vs experiment 1 (was 2e-5)
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer_exp2 = Trainer(
    model=model_exp2,
    args=training_args_exp2,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer_exp2.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.085369,0.046885,0.876803,0.901483,0.888971,0.992316
2,0.034535,0.046218,0.884701,0.896708,0.890664,0.992628
3,0.026276,0.047933,0.883806,0.908017,0.895748,0.992824


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=5619, training_loss=0.04872668962670425, metrics={'train_runtime': 1736.6175, 'train_samples_per_second': 103.518, 'train_steps_per_second': 3.236, 'total_flos': 6198235372007640.0, 'train_loss': 0.04872668962670425, 'epoch': 3.0})

In [17]:
test_results = trainer.evaluate(eval_dataset=test_ds)
print(test_results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.04144091531634331, 'eval_precision': 0.8973729437760865, 'eval_recall': 0.9089778662024373, 'eval_f1': 0.90313812700766, 'eval_accuracy': 0.993360528538596, 'eval_runtime': 22.9362, 'eval_samples_per_second': 360.217, 'eval_steps_per_second': 11.292, 'epoch': 3.0}


In [18]:
import pandas as pd

pd.DataFrame([test_results])

,eval_loss,eval_precision,eval_recall,eval_f1,eval_accuracy,eval_runtime,eval_samples_per_second,eval_steps_per_second,epoch
0,0.041441,0.897373,0.908978,0.903138,0.993361,22.9362,360.217,11.292,3.0


In [19]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

model.push_to_hub("marianaY/ner-loc-date-anonymizer")
tokenizer.push_to_hub("marianaY/ner-loc-date-anonymizer")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/marianaY/ner-loc-date-anonymizer/commit/5c0056417ba66ffb315ff7d165794e9ceffe11d8', commit_message='Upload tokenizer', commit_description='', oid='5c0056417ba66ffb315ff7d165794e9ceffe11d8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/marianaY/ner-loc-date-anonymizer', endpoint='https://huggingface.co', repo_type='model', repo_id='marianaY/ner-loc-date-anonymizer'), pr_revision=None, pr_num=None)